In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- TPU guard: CLI pushes get a plain CPU machine; the real run is started from the editor with
#     Accelerator = TPU v5e-8. Stop at once if there is no TPU (checked in a subprocess so this
#     notebook never holds the chips that the training processes need).
os.environ.pop("TPU_PROCESS_ADDRESSES", None)
probe = subprocess.run([sys.executable, "-c", "import torch_xla, torch_xla.core.xla_model as xm; "
                        "print(xm.xla_device_hw(torch_xla.device()))"], capture_output=True, text=True)
print("accelerator:", probe.stdout.strip() or probe.stderr.strip()[-300:])
if "TPU" not in probe.stdout:
    raise SystemExit("No TPU. Open this notebook in the Kaggle editor, Accelerator -> TPU v5e-8, "
                     "then Save Version -> Save & Run All.")


In [ ]:
# --- locate the tokenizer and all token-shard dirs among the attached kernel outputs ---
DATA_DIRS = inputs_with("manifest.json", "vishalam-data-")
TOKENIZER = inputs_with("tokenizer.json", "vishalam-data-1")[0] + "/tokenizer.json"
print("shard dirs:", DATA_DIRS)


In [ ]:
# --- Run a command with a watchdog and host-memory reports (a crash on the TPU VM otherwise
#     leaves no trace: a process killed for memory prints nothing) ---
import threading
def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
def host_mem():
    avail = sh("awk '/MemAvailable/ {printf \"%.0f\", $2/1048576}' /proc/meminfo")
    rss = sh("ps -eo rss,comm | awk '/python/ {s+=$1} END {printf \"%.1f\", s/1048576}'")
    peak = sh("cat /sys/fs/cgroup/memory.peak 2>/dev/null | awk '{printf \"%.0f\", $1/1073741824}'")
    return f"[mem] free {avail} GB, python procs {rss} GB, cgroup peak {peak or '?'} GB"
def run_watched(cmd, silent_min=40):
    """Stream the command's output; kill it after `silent_min` minutes without output (e.g. a stuck
    collective would otherwise burn the whole 9 h session); report memory every minute early on."""
    print("host:", sh("nproc"), "cpus,", sh("free -g | awk '/Mem/ {print $2}'"), "GB RAM, cgroup limit",
          sh("cat /sys/fs/cgroup/memory.max 2>/dev/null || cat /sys/fs/cgroup/memory/memory.limit_in_bytes"))
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    last, t_run, seen = [time.time()], time.time(), []
    def watch():
        while proc.poll() is None:
            if time.time() - last[0] > silent_min * 60:
                print(f"WATCHDOG: no output for {silent_min} min - stopping the run", flush=True)
                proc.kill()
            print(host_mem(), flush=True)
            time.sleep(60 if time.time() - t_run < 2400 else 600)
    threading.Thread(target=watch, daemon=True).start()
    for line in proc.stdout:
        last[0] = time.time()
        if "RESOURCE_EXHAUSTED" in line or "saved checkpoint" in line:
            seen.append(line)
        if not any(k in line for k in ("Warning", "warnings.warn", "symbolize")):
            print(line, end="", flush=True)
    code = proc.wait()
    print("exit code", code, "|", host_mem())
    if code != 0:  # was it the out-of-memory killer?
        print(sh("cat /sys/fs/cgroup/memory.events 2>/dev/null"))
        print(sh("dmesg -T 2>/dev/null | grep -i -E 'killed process|out of memory|oom' | tail -10")
              or "(dmesg not readable)")
    run_watched.hbm_oom = any("RESOURCE_EXHAUSTED" in l for l in seen) and not any("saved checkpoint" in l for l in seen)
    return code


In [ ]:
# --- Pretraining session on TPU v5e-8 (8 chips, bf16): resume from the latest 350m session
#     (GPU or TPU, checkpoints are interchangeable), stop cleanly before Kaggle's 9 h TPU limit ---
prev = inputs_with("run/ckpt.pt", "vishalam-pretrain-350m-")
resume = f"--resume-from {prev[0]}/run" if prev else ""
print("resuming from:", prev or "scratch")
# SPMD micro batch over 8 chips: 2 sequences per chip (4 ran out of the 16 GB HBM); if even that
# does not fit, retry at once with 1 per chip rather than lose a session that queued for hours
for micro in (16, 8):
    cmd = (f"python -m vishalam.train.pretrain --xla --config configs/vishalam-350m.yaml "
           f"--data-dirs {','.join(DATA_DIRS)} --tokenizer {TOKENIZER} {resume} --out {WORK}/run "
           f"--max-hours {hours_left(8.3):.2f} --set micro_batch={micro} --set grad_checkpointing=true "
           f"--set save_every_minutes=20")
    code = run_watched(cmd)
    if not (code and run_watched.hbm_oom):
        break
    print(f"TPU memory too small for micro_batch={micro}; retrying smaller", flush=True)
